In [ ]:
"""
MODULE CALCIFICATIONS VASCULAIRES
À coller dans un Kaggle Notebook
Dataset : VinBigData Chest X-ray Abnormalities Detection
"""
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from sklearn.metrics import classification_report, roc_auc_score
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
import json

# ============================================================
# CONFIGURATION
# ============================================================
# Remplace les 2 lignes de configuration par ceci
TRAIN_CSV = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv'
TRAIN_DIR = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train'
IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 20
LR         = 1e-4
MAX_NORMAL = 1000   # Limiter les normaux pour équilibrer
MAX_CALCIF = 452    # Toutes les images calcifications
MAX_AORTE  = 1000   # Limiter aortic enlargement

CATEGORIES = ['Normal', 'Calcification', 'Aortic_enlargement']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ============================================================
# 1. LIRE LE DICOM ET CONVERTIR EN IMAGE
# ============================================================
def lire_dicom(image_id, train_dir):
    """
    Lit un fichier DICOM et retourne une image PIL en niveaux de gris.
    Gère la normalisation des pixels DICOM.
    """
    path = os.path.join(train_dir, f"{image_id}.dicom")
    try:
        dcm     = pydicom.dcmread(path)
        img_arr = dcm.pixel_array.astype(np.float32)

        # Normalisation 0-255
        img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-6)
        img_arr = (img_arr * 255).astype(np.uint8)

        # Certains DICOM sont inversés (PhotometricInterpretation)
        if hasattr(dcm, 'PhotometricInterpretation'):
            if dcm.PhotometricInterpretation == 'MONOCHROME1':
                img_arr = 255 - img_arr

        return Image.fromarray(img_arr).convert('L')
    except Exception as e:
        return None


# ============================================================
# 2. DATASET CALCIFICATIONS
# ============================================================
class CalcificationDataset(Dataset):
    def __init__(self, csv_path, train_dir, transform=None):
        self.train_dir = train_dir
        self.transform = transform
        self.image_ids = []
        self.labels    = []
        self.bboxes    = []   # Bounding boxes pour visualisation

        df = pd.read_csv(csv_path)
        print(f"CSV chargé : {len(df)} annotations")

        # --- Classe 1 : Calcification ---
        calcif_df  = df[df['class_name'] == 'Calcification']
        calcif_ids = calcif_df['image_id'].unique()[:MAX_CALCIF]
        for img_id in calcif_ids:
            rows = calcif_df[calcif_df['image_id'] == img_id]
            bbox = rows[['x_min','y_min','x_max','y_max']].values.tolist()
            self.image_ids.append(img_id)
            self.labels.append(1)
            self.bboxes.append(bbox)

        # --- Classe 2 : Aortic enlargement ---
        aorte_df  = df[df['class_name'] == 'Aortic enlargement']
        aorte_ids = aorte_df['image_id'].unique()[:MAX_AORTE]
        for img_id in aorte_ids:
            rows = aorte_df[aorte_df['image_id'] == img_id]
            bbox = rows[['x_min','y_min','x_max','y_max']].values.tolist()
            self.image_ids.append(img_id)
            self.labels.append(2)
            self.bboxes.append(bbox)

        # --- Classe 0 : Normal ---
        normal_ids = df[df['class_name'] == 'No finding']['image_id'].unique()[:MAX_NORMAL]
        for img_id in normal_ids:
            self.image_ids.append(img_id)
            self.labels.append(0)
            self.bboxes.append([])

        # Résumé
        counts = Counter(self.labels)
        print(f"\nDataset calcifications :")
        for i, cat in enumerate(CATEGORIES):
            print(f"  {cat:<22}: {counts.get(i,0)} images")
        print(f"  Total            : {len(self.image_ids)} images")

        # Poids pour WeightedRandomSampler
        total = len(self.labels)
        self.sample_weights = [
            total / (counts.get(l, 1) * len(CATEGORIES))
            for l in self.labels
        ]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        label  = self.labels[idx]
        img    = lire_dicom(img_id, self.train_dir)

        if img is None:
            # Image corrompue → image noire
            img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8))

        if self.transform:
            img = self.transform(img)

        return img, label


# ============================================================
# 3. CNN EfficientNet pour calcifications
# ============================================================
class CalcifCNN(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super(CalcifCNN, self).__init__()

        self.backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        )

        # Adapter RGB → Grayscale
        orig = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            1, orig.out_channels,
            kernel_size=orig.kernel_size,
            stride=orig.stride,
            padding=orig.padding,
            bias=False
        )
        with torch.no_grad():
            self.backbone.features[0][0].weight = nn.Parameter(
                orig.weight.mean(dim=1, keepdim=True)
            )

        # Tête de classification
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(256, num_classes)
        )

        # Hook pour Grad-CAM
        self._activations = None
        self.backbone.features[-1].register_forward_hook(self._save_activation)

    def _save_activation(self, module, input, output):
        self._activations = output

    def forward(self, x):
        return self.backbone(x)

    def get_gradcam(self, input_tensor):
        self.eval()
        input_tensor = input_tensor.clone().requires_grad_(True)
        self._activations = None
        output = self.backbone(input_tensor)
        target = output.argmax(dim=1).item()
        self.zero_grad()
        output[0, target].backward()
        if self._activations is None:
            return None, output
        act     = self._activations.detach().cpu().numpy()[0]
        weights = act.mean(axis=(1, 2))
        cam     = sum(w * a for w, a in zip(weights, act))
        cam     = cam.clip(min=0)
        if cam.max() > 0:
            cam /= cam.max()
        return cam, output


# ============================================================
# 4. ENTRAÎNEMENT
# ============================================================
def train_calcif_model():
    t_train = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])
    t_val = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    dataset = CalcificationDataset(TRAIN_CSV, TRAIN_DIR, transform=t_train)
    n       = len(dataset)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)
    n_test  = n - n_train - n_val

    train_set, val_set, test_set = random_split(
        dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )
    val_set.dataset.transform  = t_val
    test_set.dataset.transform = t_val

    print(f"\nSplit : Train={n_train} | Val={n_val} | Test={n_test}")

    # WeightedSampler
    train_w = [dataset.sample_weights[i] for i in train_set.indices]
    sampler  = torch.utils.data.WeightedRandomSampler(
        train_w, num_samples=len(train_w), replacement=True
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                              sampler=sampler, num_workers=2)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2)

    # Class weights
    counts = [dataset.labels.count(i) for i in range(len(CATEGORIES))]
    total  = sum(counts)
    cw     = torch.tensor([total / (c * len(CATEGORIES)) for c in counts],
                          dtype=torch.float).to(device)
    print(f"Class weights : { {cat: round(w,2) for cat,w in zip(CATEGORIES,cw.tolist())} }")

    model     = CalcifCNN(num_classes=len(CATEGORIES)).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0
    historique   = []

    print(f"\n{'Époque':<8} {'Loss':<12} {'Train Acc':<12} {'Val Acc':<10}")
    print("-" * 44)

    for epoch in range(EPOCHS):
        # TRAIN
        model.train()
        t_loss, correct, total_s = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            t_loss  += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total_s += labels.size(0)

        train_acc = correct / total_s

        # VALIDATION
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out       = model(imgs)
                v_correct += (out.argmax(1) == labels).sum().item()
                v_total   += labels.size(0)

        val_acc = v_correct / v_total
        scheduler.step()

        historique.append({
            'epoch': epoch+1,
            'train_loss': round(t_loss/len(train_loader), 4),
            'train_acc':  round(train_acc, 4),
            'val_acc':    round(val_acc, 4)
        })

        marker = "  ← meilleur" if val_acc > best_val_acc else ""
        print(f"{epoch+1:<8} {t_loss/len(train_loader):.4f}{'':7} "
              f"{train_acc:.3f}{'':7} {val_acc:.3f}{marker}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'calcif_model_best.pth')

    torch.save(model.state_dict(), 'calcif_model_final.pth')
    with open('historique_calcif.json', 'w') as f:
        json.dump(historique, f, indent=2)

    print(f"\nMeilleure Val Acc : {best_val_acc:.3f}")

    # --- Évaluation test set ---
    model.load_state_dict(torch.load('calcif_model_best.pth'))
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            output = model(imgs)
            probs  = torch.softmax(output, dim=1).cpu().numpy()
            preds  = output.argmax(1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print("\n=== RAPPORT TEST SET ===")
    print(classification_report(all_labels, all_preds,
                                 target_names=CATEGORIES, digits=3))

    # Courbes d'apprentissage
    eps    = [x['epoch']      for x in historique]
    losses = [x['train_loss'] for x in historique]
    t_accs = [x['train_acc']  for x in historique]
    v_accs = [x['val_acc']    for x in historique]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(eps, losses, 'b-o', markersize=3)
    axes[0].set_title('Loss', fontweight='bold')
    axes[0].set_xlabel('Époques'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(eps, t_accs, 'b-o', markersize=3, label='Train')
    axes[1].plot(eps, v_accs, 'r-o', markersize=3, label='Val')
    axes[1].set_title('Accuracy', fontweight='bold')
    axes[1].set_xlabel('Époques')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('Entraînement — Module Calcifications Vasculaires',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('courbes_calcif.png', dpi=150, bbox_inches='tight')
    plt.show()

    return model, test_loader


# ============================================================
# EXÉCUTION
# ============================================================
model, test_loader = train_calcif_model()

print("\nFichiers générés :")
print("  calcif_model_best.pth  ← à télécharger et uploader sur Drive")
print("  calcif_model_final.pth")
print("  historique_calcif.json")
print("  courbes_calcif.png")